# Bluestock Mutual Fund Analytics

## 04 - Performance Analytics

This notebook calculates and analyses mutual fund performance and risk
metrics using historical NAV and benchmark data.

Metrics include CAGR, volatility, Sharpe ratio, Beta, Value at Risk (VaR),
and maximum drawdown.

In [29]:
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

PROJECT_ROOT = Path.cwd().parent
PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"

print("Processed folder:", PROCESSED_DIR)

Processed folder: c:\Users\Farhan\bluestock_mf_capstone\data\processed


In [30]:
def find_dataset(keyword):
    files = list(PROCESSED_DIR.glob(f"*{keyword}*.csv"))
    
    if not files:
        raise FileNotFoundError(
            f"Dataset not found: {keyword}"
        )
    
    return files[0]


nav_file = find_dataset("nav_history")
fund_file = find_dataset("fund_master")
benchmark_file = find_dataset("benchmark_indices")

nav = pd.read_csv(nav_file)
funds = pd.read_csv(fund_file)
benchmark = pd.read_csv(benchmark_file)

print("NAV:", nav.shape)
print("Fund Master:", funds.shape)
print("Benchmark:", benchmark.shape)

NAV: (46000, 3)
Fund Master: (40, 15)
Benchmark: (8050, 3)


In [31]:
nav["date"] = pd.to_datetime(
    nav["date"],
    errors="coerce"
)

nav["nav"] = pd.to_numeric(
    nav["nav"],
    errors="coerce"
)

nav = nav.dropna(
    subset=["amfi_code", "date", "nav"]
)

nav = nav.sort_values(
    ["amfi_code", "date"]
)

nav["daily_return"] = (
    nav.groupby("amfi_code")["nav"]
       .pct_change()
)

nav = nav.dropna(
    subset=["daily_return"]
)

print("Return calculation completed.")
display(nav.head())

Return calculation completed.


,amfi_code,date,nav,daily_return
5751,100016,2022-01-04,515.0971,-0.010306
5752,100016,2022-01-05,521.7239,0.012865
5753,100016,2022-01-06,515.7880,-0.011377
5754,100016,2022-01-07,515.1639,-0.001210
5755,100016,2022-01-10,510.7136,-0.008639


In [32]:
cagr_results = []

for amfi_code, group in nav.groupby("amfi_code"):

    group = group.sort_values("date")

    start_nav = group.iloc[0]["nav"]
    end_nav = group.iloc[-1]["nav"]

    start_date = group.iloc[0]["date"]
    end_date = group.iloc[-1]["date"]

    years = (
        end_date - start_date
    ).days / 365.25

    if years > 0 and start_nav > 0:
        cagr = (
            (end_nav / start_nav) ** (1 / years) - 1
        ) * 100
    else:
        cagr = np.nan

    cagr_results.append({
        "amfi_code": int(amfi_code),
        "cagr_pct": cagr
    })

cagr_df = pd.DataFrame(cagr_results)

cagr_df = cagr_df.merge(
    funds[
        ["amfi_code", "scheme_name"]
    ],
    on="amfi_code",
    how="left"
)

display(
    cagr_df.sort_values(
        "cagr_pct",
        ascending=False
    ).head(10)
)

,amfi_code,cagr_pct,scheme_name
25,120505,33.459588,ICICI Pru Midcap Fund - Regular - Growth
21,119598,32.894606,SBI Small Cap Fund - Regular Plan - Growth
39,149324,32.628442,DSP Small Cap Fund - Regular - Growth
36,148569,31.722136,Mirae Asset Tax Saver Fund - Regular - Growth
2,100033,30.542236,HDFC Mid-Cap Opportunities Fund - Regular - Gr...
34,148567,30.427322,Mirae Asset Large Cap Fund - Regular - Growth
30,120843,30.394617,Kotak Flexicap Fund - Regular - Growth
38,149323,29.587801,DSP Midcap Fund - Regular - Growth
16,119094,28.339403,Axis Midcap Fund - Regular - Growth
19,119551,25.842778,SBI Bluechip Fund - Regular Plan - Growth


In [33]:
volatility = (
    nav.groupby("amfi_code")["daily_return"]
       .std()
       .mul(np.sqrt(252))
       .mul(100)
       .reset_index(name="volatility_pct")
)

volatility = volatility.merge(
    funds[
        ["amfi_code", "scheme_name"]
    ],
    on="amfi_code",
    how="left"
)

display(
    volatility.sort_values(
        "volatility_pct",
        ascending=False
    ).head(10)
)

,amfi_code,volatility_pct,scheme_name
4,101207,25.797322,ABSL Small Cap Fund - Regular - Growth
11,118634,25.241521,Nippon India Small Cap Fund - Regular - Growth
21,119598,25.140579,SBI Small Cap Fund - Regular Plan - Growth
17,119095,25.066580,Axis Small Cap Fund - Regular - Growth
22,119599,24.950125,SBI Small Cap Fund - Direct Plan - Growth
39,149324,24.840205,DSP Small Cap Fund - Regular - Growth
16,119094,19.407117,Axis Midcap Fund - Regular - Growth
25,120505,19.290949,ICICI Pru Midcap Fund - Regular - Growth
2,100033,18.936711,HDFC Mid-Cap Opportunities Fund - Regular - Gr...
33,125498,18.352430,HDFC Mid-Cap Opportunities Fund - Direct - Growth


In [34]:
drawdown_results = []

for amfi_code, group in nav.groupby("amfi_code"):

    group = group.sort_values("date").copy()

    running_max = group["nav"].cummax()

    drawdown = (
        group["nav"] / running_max - 1
    )

    max_drawdown = drawdown.min() * 100

    drawdown_results.append({
        "amfi_code": int(amfi_code),
        "max_drawdown_pct": max_drawdown
    })

drawdown_df = pd.DataFrame(drawdown_results)

drawdown_df = drawdown_df.merge(
    funds[
        ["amfi_code", "scheme_name"]
    ],
    on="amfi_code",
    how="left"
)

display(
    drawdown_df.sort_values(
        "max_drawdown_pct"
    ).head(10)
)

,amfi_code,max_drawdown_pct,scheme_name
22,119599,-52.574221,SBI Small Cap Fund - Direct Plan - Growth
17,119095,-51.677754,Axis Small Cap Fund - Regular - Growth
4,101207,-35.446916,ABSL Small Cap Fund - Regular - Growth
39,149324,-31.171900,DSP Small Cap Fund - Regular - Growth
21,119598,-28.706006,SBI Small Cap Fund - Regular Plan - Growth
7,102886,-28.001124,UTI Mid Cap Fund - Regular - Growth
0,100016,-24.734441,HDFC Top 100 Fund - Regular Plan - Growth
29,120842,-24.003511,Kotak Emerging Equity Fund - Regular - Growth
11,118634,-23.344886,Nippon India Small Cap Fund - Regular - Growth
15,119093,-21.751396,Axis Bluechip Fund - Direct - Growth


In [35]:
var_results = (
    nav.groupby("amfi_code")["daily_return"]
       .quantile(0.05)
       .mul(100)
       .reset_index(name="VaR_95_daily_pct")
)

var_results = var_results.merge(
    funds[
        ["amfi_code", "scheme_name"]
    ],
    on="amfi_code",
    how="left"
)

display(var_results.head(10))

,amfi_code,VaR_95_daily_pct,scheme_name
0,100016,-1.436364,HDFC Top 100 Fund - Regular Plan - Growth
1,100025,-0.379325,HDFC Short Term Debt Fund - Regular - Growth
2,100033,-1.903354,HDFC Mid-Cap Opportunities Fund - Regular - Gr...
3,101206,-1.328166,ABSL Frontline Equity Fund - Regular - Growth
4,101207,-2.602125,ABSL Small Cap Fund - Regular - Growth
5,101208,-0.026866,ABSL Liquid Fund - Regular - Growth
6,102885,-1.261263,UTI Nifty 50 Index Fund - Regular - Growth
7,102886,-1.922028,UTI Mid Cap Fund - Regular - Growth
8,102887,-1.523162,UTI Flexi Cap Fund - Regular - Growth
9,118632,-1.395405,Nippon India Large Cap Fund - Regular - Growth


In [36]:
performance_metrics = (
    cagr_df[
        ["amfi_code", "scheme_name", "cagr_pct"]
    ]
    .merge(
        volatility[
            ["amfi_code", "volatility_pct"]
        ],
        on="amfi_code",
        how="left"
    )
    .merge(
        drawdown_df[
            ["amfi_code", "max_drawdown_pct"]
        ],
        on="amfi_code",
        how="left"
    )
    .merge(
        var_results[
            ["amfi_code", "VaR_95_daily_pct"]
        ],
        on="amfi_code",
        how="left"
    )
)

display(
    performance_metrics.sort_values(
        "cagr_pct",
        ascending=False
    ).head(10)
)

,amfi_code,scheme_name,cagr_pct,volatility_pct,max_drawdown_pct,VaR_95_daily_pct
25,120505,ICICI Pru Midcap Fund - Regular - Growth,33.459588,19.290949,-18.188514,-1.889179
21,119598,SBI Small Cap Fund - Regular Plan - Growth,32.894606,25.140579,-28.706006,-2.450705
39,149324,DSP Small Cap Fund - Regular - Growth,32.628442,24.840205,-31.171900,-2.348307
36,148569,Mirae Asset Tax Saver Fund - Regular - Growth,31.722136,17.674007,-16.396743,-1.711819
2,100033,HDFC Mid-Cap Opportunities Fund - Regular - Gr...,30.542236,18.936711,-16.217209,-1.903354
34,148567,Mirae Asset Large Cap Fund - Regular - Growth,30.427322,14.193707,-11.265729,-1.355978
30,120843,Kotak Flexicap Fund - Regular - Growth,30.394617,15.886987,-12.973968,-1.450794
38,149323,DSP Midcap Fund - Regular - Growth,29.587801,17.746159,-17.248106,-1.788208
16,119094,Axis Midcap Fund - Regular - Growth,28.339403,19.407117,-20.960884,-1.848028
19,119551,SBI Bluechip Fund - Regular Plan - Growth,25.842778,13.741434,-15.012385,-1.284623


In [37]:
benchmark["date"] = pd.to_datetime(
    benchmark["date"],
    errors="coerce"
)

benchmark["close_value"] = pd.to_numeric(
    benchmark["close_value"],
    errors="coerce"
)

benchmark = benchmark.dropna(
    subset=["date", "close_value"]
)

benchmark = benchmark.sort_values("date")

benchmark["benchmark_return"] = (
    benchmark.groupby("index_name")["close_value"]
             .pct_change()
)

benchmark = benchmark.dropna(
    subset=["benchmark_return"]
)

print("Benchmark data prepared.")
display(benchmark.head())

Benchmark data prepared.


,date,index_name,close_value,benchmark_return
1,2022-01-04,NIFTY50,17689.64,0.011253
5751,2022-01-04,CRISIL_LIQUID,2281.61,0.000044
6901,2022-01-04,CRISIL_GILT,1453.26,0.001516
1151,2022-01-04,NIFTY100,17537.52,-0.013540
2301,2022-01-04,NIFTY_MIDCAP150,9954.23,0.023909


In [38]:
print("Available benchmark indices:")

print(
    benchmark["index_name"]
    .dropna()
    .unique()
)

Available benchmark indices:
<StringArray>
[        'NIFTY50',   'CRISIL_LIQUID',     'CRISIL_GILT',        'NIFTY100',
 'NIFTY_MIDCAP150',    'BSE_SMALLCAP',        'NIFTY500']
Length: 7, dtype: str


In [39]:
# Use NIFTY100 as the benchmark
benchmark_name = "NIFTY100"

benchmark_selected = benchmark[
    benchmark["index_name"] == benchmark_name
].copy()

print("Benchmark selected:", benchmark_name)
print("Benchmark rows:", len(benchmark_selected))

display(benchmark_selected.head())

Benchmark selected: NIFTY100
Benchmark rows: 1149


,date,index_name,close_value,benchmark_return
1151,2022-01-04,NIFTY100,17537.52,-0.013540
1152,2022-01-05,NIFTY100,17607.73,0.004003
1153,2022-01-06,NIFTY100,17556.05,-0.002935
1154,2022-01-07,NIFTY100,17664.02,0.006150
1155,2022-01-10,NIFTY100,17516.51,-0.008351


In [40]:
# Prepare fund daily returns
fund_returns = nav[
    ["amfi_code", "date", "daily_return"]
].copy()

fund_returns = fund_returns.rename(
    columns={"daily_return": "fund_return"}
)

# Prepare benchmark returns
benchmark_returns = benchmark_selected[
    ["date", "benchmark_return"]
].copy()

# Match fund returns with benchmark returns on the same date
merged_returns = fund_returns.merge(
    benchmark_returns,
    on="date",
    how="inner"
)

print("Aligned observations:", len(merged_returns))

display(merged_returns.head())

Aligned observations: 45960


,amfi_code,date,fund_return,benchmark_return
0,100016,2022-01-04,-0.010306,-0.013540
1,100016,2022-01-05,0.012865,0.004003
2,100016,2022-01-06,-0.011377,-0.002935
3,100016,2022-01-07,-0.001210,0.006150
4,100016,2022-01-10,-0.008639,-0.008351


In [41]:
beta_results = []

for amfi_code, group in merged_returns.groupby("amfi_code"):

    group = group.dropna(
        subset=["fund_return", "benchmark_return"]
    )

    if len(group) > 1:
        benchmark_variance = group["benchmark_return"].var()

        if benchmark_variance > 0:
            beta = (
                group["fund_return"].cov(
                    group["benchmark_return"]
                )
                / benchmark_variance
            )
        else:
            beta = np.nan
    else:
        beta = np.nan

    beta_results.append({
        "amfi_code": amfi_code,
        "beta": beta
    })

beta_df = pd.DataFrame(beta_results)

beta_df = beta_df.merge(
    funds[
        ["amfi_code", "scheme_name"]
    ],
    on="amfi_code",
    how="left"
)

display(
    beta_df.sort_values(
        "beta",
        ascending=False
    ).head(10)
)

,amfi_code,beta,scheme_name
11,118634,0.103497,Nippon India Small Cap Fund - Regular - Growth
22,119599,0.062002,SBI Small Cap Fund - Direct Plan - Growth
32,125497,0.048820,HDFC Top 100 Fund - Direct Plan - Growth
26,120506,0.041896,ICICI Pru Value Discovery Fund - Regular - Growth
28,120841,0.036356,Kotak Bluechip Fund - Regular - Growth
15,119093,0.025883,Axis Bluechip Fund - Direct - Growth
34,148567,0.023684,Mirae Asset Large Cap Fund - Regular - Growth
3,101206,0.021086,ABSL Frontline Equity Fund - Regular - Growth
36,148569,0.018134,Mirae Asset Tax Saver Fund - Regular - Growth
29,120842,0.018057,Kotak Emerging Equity Fund - Regular - Growth


In [42]:
sharpe_results = []

for amfi_code, group in nav.groupby("amfi_code"):

    returns = group["daily_return"].dropna()

    if len(returns) > 1 and returns.std() > 0:
        sharpe = (
            returns.mean()
            / returns.std()
        ) * np.sqrt(252)
    else:
        sharpe = np.nan

    sharpe_results.append({
        "amfi_code": amfi_code,
        "sharpe_ratio": sharpe
    })

sharpe_df = pd.DataFrame(sharpe_results)

sharpe_df = sharpe_df.merge(
    funds[
        ["amfi_code", "scheme_name"]
    ],
    on="amfi_code",
    how="left"
)

display(
    sharpe_df.sort_values(
        "sharpe_ratio",
        ascending=False
    ).head(10)
)

,amfi_code,sharpe_ratio,scheme_name
27,120507,13.655946,ICICI Pru Liquid Fund - Regular - Growth
31,120844,12.573992,Kotak Liquid Fund - Regular - Growth
5,101208,12.019194,ABSL Liquid Fund - Regular - Growth
34,148567,1.906241,Mirae Asset Large Cap Fund - Regular - Growth
30,120843,1.715884,Kotak Flexicap Fund - Regular - Growth
19,119551,1.681289,SBI Bluechip Fund - Regular Plan - Growth
36,148569,1.602702,Mirae Asset Tax Saver Fund - Regular - Growth
9,118632,1.541077,Nippon India Large Cap Fund - Regular - Growth
25,120505,1.517047,ICICI Pru Midcap Fund - Regular - Growth
38,149323,1.498398,DSP Midcap Fund - Regular - Growth


In [43]:
performance_final = (
    performance_metrics
    .merge(
        beta_df[
            ["amfi_code", "beta"]
        ],
        on="amfi_code",
        how="left"
    )
    .merge(
        sharpe_df[
            ["amfi_code", "sharpe_ratio"]
        ],
        on="amfi_code",
        how="left"
    )
)

display(
    performance_final.sort_values(
        "cagr_pct",
        ascending=False
    ).head(10)
)

,amfi_code,scheme_name,cagr_pct,volatility_pct,max_drawdown_pct,VaR_95_daily_pct,beta,sharpe_ratio
25,120505,ICICI Pru Midcap Fund - Regular - Growth,33.459588,19.290949,-18.188514,-1.889179,0.000549,1.517047
21,119598,SBI Small Cap Fund - Regular Plan - Growth,32.894606,25.140579,-28.706006,-2.450705,-0.023196,1.203854
39,149324,DSP Small Cap Fund - Regular - Growth,32.628442,24.840205,-31.171900,-2.348307,0.011455,1.211468
36,148569,Mirae Asset Tax Saver Fund - Regular - Growth,31.722136,17.674007,-16.396743,-1.711819,0.018134,1.602702
2,100033,HDFC Mid-Cap Opportunities Fund - Regular - Gr...,30.542236,18.936711,-16.217209,-1.903354,0.005104,1.436947
34,148567,Mirae Asset Large Cap Fund - Regular - Growth,30.427322,14.193707,-11.265729,-1.355978,0.023684,1.906241
30,120843,Kotak Flexicap Fund - Regular - Growth,30.394617,15.886987,-12.973968,-1.450794,-0.022830,1.715884
38,149323,DSP Midcap Fund - Regular - Growth,29.587801,17.746159,-17.248106,-1.788208,-0.002523,1.498398
16,119094,Axis Midcap Fund - Regular - Growth,28.339403,19.407117,-20.960884,-1.848028,-0.066265,1.333160
19,119551,SBI Bluechip Fund - Regular Plan - Growth,25.842778,13.741434,-15.012385,-1.284623,-0.031751,1.681289


In [44]:
output_file = (
    PROCESSED_DIR /
    "performance_metrics_calculated.csv"
)

performance_final.to_csv(
    output_file,
    index=False
)

print("Saved:", output_file)

Saved: c:\Users\Farhan\bluestock_mf_capstone\data\processed\performance_metrics_calculated.csv


## Performance Analytics Methodology

- CAGR is calculated using the actual elapsed time between the first and last available NAV observations.
- Annualised volatility is calculated from daily returns using sqrt(252).
- Maximum drawdown is calculated from the historical NAV peak-to-trough decline.
- Daily 95% historical VaR is estimated using the 5th percentile of daily returns.
- Beta is calculated using covariance of fund and NIFTY100 daily returns divided by benchmark return variance.
- Sharpe ratio is annualised using daily returns and assumes a zero daily risk-free rate because a risk-free-rate time series is not available in the provided datasets.